In [ ]:
"""
Outfitly - Feature Engineering Script
======================================
Prepares the data for the Two-Tower model.
- Label encodes categorical features (Category, Style Tags).
- Normalizes numerical features (Price).
- Generates user index and item index mapping.
- Computes baseline features for Users (e.g. Average Purchase Price).
- Saves processed datasets to CSV for fast loading in model.py.
"""

In [ ]:
import os
import pickle
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sqlalchemy import create_engine

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
SCRIPT_DIR = os.getcwd()
DATA_DIR = os.path.join(SCRIPT_DIR, "data")
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
CONNECTION_STRING = (
    "mssql+pyodbc://@(localdb)\\mssqllocaldb/Outfitly"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&Trusted_Connection=yes"
    "&TrustServerCertificate=yes"
)

In [ ]:
def get_engine():
    return create_engine(CONNECTION_STRING)

In [ ]:
def main():
    print("=" * 60)
    print("Outfitly - Feature Engineering")
    print("=" * 60)
    
    engine = get_engine()

    # 1. Load Data
    print("[1/5] Loading data from SQL Server and CSV...")
    with engine.connect() as conn:
        df_products = pd.read_sql("SELECT Id, Category, Price FROM Products", conn)
        df_interactions = pd.read_sql("SELECT UserId, ProductId, Price FROM UserInteractions", conn)
    
    style_tags_path = os.path.join(DATA_DIR, "style_tags.csv")
    if os.path.exists(style_tags_path):
        df_tags = pd.read_csv(style_tags_path)
    else:
        print("ERROR: style_tags.csv not found! Did you run data_preparation.py?")
        return

    # 2. Process Item Features
    print("[2/5] Processing Item features...")
    # Merge Products with Style Tags
    df_items = df_products.merge(df_tags, on="Id", how="left")
    
    # Fill NA for missing categories or tags
    df_items["Category"] = df_items["Category"].fillna("Unknown")
    df_items["colour_group_name"] = df_items["colour_group_name"].fillna("Unknown")
    df_items["graphical_appearance_name"] = df_items["graphical_appearance_name"].fillna("Unknown")
    
    # Label Encoders
    item_le = LabelEncoder()
    cat_le = LabelEncoder()
    color_le = LabelEncoder()
    graphic_le = LabelEncoder()
    
    df_items["ItemIndex"] = item_le.fit_transform(df_items["Id"])
    df_items["CategoryIdx"] = cat_le.fit_transform(df_items["Category"])
    df_items["ColorIdx"] = color_le.fit_transform(df_items["colour_group_name"])
    df_items["GraphicIdx"] = graphic_le.fit_transform(df_items["graphical_appearance_name"])
    
    price_scaler = MinMaxScaler()
    df_items["NormPrice"] = price_scaler.fit_transform(df_items[["Price"]].fillna(0))

    # 3. Process User Features
    print("[3/5] Processing User features...")
    user_le = LabelEncoder()
    # Calculate avg price for each user from interactions
    user_agg = df_interactions.groupby("UserId", as_index=False)["Price"].mean().rename(columns={"Price": "AvgPurchasePrice"})
    
    # Unique users that actually interacted
    unique_users = pd.DataFrame({"UserId": df_interactions["UserId"].unique()})
    unique_users = unique_users.merge(user_agg, on="UserId", how="left")
    unique_users["AvgPurchasePrice"] = unique_users["AvgPurchasePrice"].fillna(0)
    
    unique_users["UserIndex"] = user_le.fit_transform(unique_users["UserId"])
    user_price_scaler = MinMaxScaler()
    unique_users["NormAvgPrice"] = user_price_scaler.fit_transform(unique_users[["AvgPurchasePrice"]])

    # 4. Map Interactions to Indices
    print("[4/5] Mapping Interactions...")
    df_int_mapped = df_interactions[["UserId", "ProductId"]].copy()
    
    # Needs tight merge on the original strings/ints
    df_int_mapped = df_int_mapped.merge(unique_users[["UserId", "UserIndex"]], on="UserId", how="inner")
    df_int_mapped = df_int_mapped.merge(df_items[["Id", "ItemIndex"]].rename(columns={"Id": "ProductId"}), on="ProductId", how="inner")
    
    # Add label = 1 for positive samples
    df_int_mapped["Label"] = 1
    
    # Keep only what we need for training
    train_data = df_int_mapped[["UserIndex", "ItemIndex", "Label"]]

    # 5. Save Outputs
    print("[5/5] Saving processed features and encoders...")
    
    item_features = df_items[["ItemIndex", "Id", "CategoryIdx", "ColorIdx", "GraphicIdx", "NormPrice"]]
    item_features.to_csv(os.path.join(DATA_DIR, "item_features.csv"), index=False)
    
    user_features = unique_users[["UserIndex", "UserId", "NormAvgPrice"]]
    user_features.to_csv(os.path.join(DATA_DIR, "user_features.csv"), index=False)
    
    train_data.to_csv(os.path.join(DATA_DIR, "train_interactions.csv"), index=False)
    
    # Save label encoders for Retrieval API later
    encoders = {
        "item_le": item_le,
        "cat_le": cat_le,
        "color_le": color_le,
        "graphic_le": graphic_le,
        "user_le": user_le
    }
    with open(os.path.join(DATA_DIR, "encoders.pkl"), "wb") as f:
        pickle.dump(encoders, f)
        
    print(f"  Items processed: {len(item_features):,}")
    print(f"  Users processed: {len(user_features):,}")
    print(f"  Pos interactions: {len(train_data):,}")
    
    print("\n" + "=" * 60)
    print("Feature Engineering complete!")
    print("=" * 60)

In [ ]:
if __name__ == "__main__":
    main()